# GRACE H1 — Colab CPU preparation

**Điểm bắt đầu vẫn là notebooks/full-pipeline.ipynb trên máy local.** Sang notebook này sau khi đã train head trên Kaggle và có grace-h1-trained-heads.zip.

Runtime → Change runtime type → Hardware accelerator: **None (CPU)**. Notebook chỉ chạy whitening/projection bằng NumPy, candidate retrieval, Joern cho graph thiếu, reranking và đóng băng prompt. Không train encoder/head, không gọi LLM; không cần GPU, Vertex hay ADC.

Đặt ba file vào một thư mục Google Drive, ví dụ **MyDrive/GRACE-H1-transfer-reveal** (riêng cho ReVeal; dùng runtime mới):

1. grace-h1-cpu-inputs.zip — dataset, sáu embedding sets, candidate/graph caches và code/config.
2. grace-h1-joern-runtime.zip — đúng Joern 4.0.623, chỉ frontend C/C++ cần dùng.
3. grace-h1-trained-heads.zip — kết quả Kaggle phù hợp config/source/data.

Input CPU có cả query code/labels để giữ snapshot/checksum; prepare loại query labels trước retrieval. Không dùng label query để chọn demo. Tất cả file/runtime chạy trong /content; Drive chỉ giữ zip đầu vào/đầu ra để tránh I/O hàng nghìn file.

In [ ]:
from pathlib import Path,PurePosixPath
import os,sys,json,zipfile,hashlib,shutil,subprocess
USE_GOOGLE_DRIVE=True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TRANSFER=Path('/content/drive/MyDrive/GRACE-H1-transfer-reveal')
else:
    # Upload the three zip files using Colab's Files pane, then use this folder.
    TRANSFER=Path('/content/transfer-reveal')
TRANSFER.mkdir(parents=True,exist_ok=True)
ROOT=Path('/content/GRACE-H1-SecurityProjection-reveal')
INPUTS=TRANSFER/'grace-h1-cpu-inputs.zip'
HEADS=TRANSFER/'grace-h1-trained-heads.zip'
JOERN=TRANSFER/'grace-h1-joern-runtime.zip'
CHECKPOINT=TRANSFER/'grace-h1-cpu-checkpoint.zip'
OUTPUT=TRANSFER/'grace-h1-cpu-prepared.zip'
for p in (INPUTS,HEADS,JOERN):
    if not p.is_file(): raise FileNotFoundError(f'Upload required bundle first: {p}')
print('Transfer:',TRANSFER,'CPU RAM GiB:',round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1024**3,2))

## 1. Nhập snapshot và head

Bootstrap kiểm tra đường dẫn, checksum và mode trước khi dùng code. Trên cùng runtime, file trùng được reuse; file khác nội dung bị chặn để tránh trộn experiment. Khi đổi config/source, dùng runtime mới và bundle input mới.

In [ ]:
ROOT.mkdir(parents=True,exist_ok=True)
# Copy a single archive to native disk before extracting thousands of members.
local_inputs=Path('/content/grace-h1-cpu-inputs.zip')
shutil.copyfile(INPUTS,local_inputs)
with zipfile.ZipFile(local_inputs) as z:
    manifest=json.loads(z.read('bundle_manifest.json'))
    if manifest['schema']!='grace_h1_bundle_v1' or manifest['kind']!='cpu_inputs': raise ValueError('Wrong CPU inputs')
    names=[e['path'] for e in manifest['entries']]
    if len({n.casefold() for n in names})!=len(names): raise ValueError('Duplicate paths')
    if sorted(z.namelist())!=sorted(names+['bundle_manifest.json']): raise ValueError('Unexpected members')
    for e in manifest['entries']:
        name=e['path'];rel=PurePosixPath(name)
        if rel.is_absolute() or '..' in rel.parts or ':' in name or chr(92) in name: raise ValueError('Unsafe path')
        target=ROOT/name;target.resolve().relative_to(ROOT)
        if target.exists():
            h=hashlib.sha256()
            with target.open('rb') as src:
                for chunk in iter(lambda:src.read(1024*1024),b''): h.update(chunk)
            if target.stat().st_size==e['bytes'] and h.hexdigest()==e['sha256']: continue
            raise ValueError(f'Different existing input: {name}; use a new runtime')
        target.parent.mkdir(parents=True,exist_ok=True)
        temp=target.with_name(target.name+'.bundle.tmp');h=hashlib.sha256();size=0
        with z.open(name) as src,temp.open('wb') as dst:
            for chunk in iter(lambda:src.read(1024*1024),b''):
                h.update(chunk);size+=len(chunk);dst.write(chunk)
        if size!=e['bytes'] or h.hexdigest()!=e['sha256']: raise ValueError('Input checksum mismatch')
        temp.chmod(e.get('mode',0o644) & 0o777);os.replace(temp,target)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from grace_h1.portable import import_bundle,create_bundle
from grace_h1.io import read_json
print(import_bundle(ROOT,HEADS,'trained_heads'))
print(import_bundle(ROOT,JOERN,'joern_runtime'))
if CHECKPOINT.exists(): print(import_bundle(ROOT,CHECKPOINT,'cpu_checkpoint'))
print('Inputs, heads and optional checkpoint imported')

## 2. Dependency CPU và Java

Chỉ cài requirements-cpu.txt: không tải PyTorch/CUDA/CodeT5 hay SDK gọi LLM. Cài Java 21 như môi trường Joern nguồn. JOERN_HOME là biến môi trường; giữ pipeline.joern_home=null trong config để fingerprint không phụ thuộc đường dẫn máy.

Cấu hình hiện dùng 3 graph_workers, heap tối đa mỗi JVM 2 GB. Nếu cần đổi workers, sửa config ở local và tạo lại input trước khi chạy cả hai arm. Colab không bảo đảm lượng RAM/thời gian cố định. Bước graph hiện có progress bar: tổng graph đã xử lý, cache hit, graph đã parse mới, graph đang parse (kèm record ID) và ETA. Checkpoint zip lưu theo mốc 500 graph khi có graph mới và cuối mỗi arm.

In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-cpu.txt')])
subprocess.check_call(['apt-get','update','-qq'])
subprocess.check_call(['apt-get','install','-y','-qq','openjdk-21-jre-headless'])
java_homes=sorted(Path('/usr/lib/jvm').glob('java-21-openjdk-*'))
if not java_homes: raise RuntimeError('Java 21 install did not provide JAVA_HOME')
os.environ['JAVA_HOME']=str(java_homes[0])
os.environ['PATH']=str(java_homes[0]/'bin')+os.pathsep+os.environ['PATH']
os.environ['JOERN_HOME']=str(ROOT/'tools'/'joern'/'joern-cli')
subprocess.check_call([str(java_homes[0]/'bin'/'java'),'-version'])
config=read_json(ROOT/'configs'/'experiment.json')
if config['datasets'] != ['reveal']: raise ValueError('Cần CPU-inputs mới có datasets=[reveal]; dùng runtime mới')
if config['pipeline']['joern_home'] is not None: raise ValueError('Set pipeline.joern_home=null in the local config and recreate CPU inputs')
from grace_h1.offload import require_trained_heads
heads=require_trained_heads(ROOT,config)
print('Verified heads; datasets:',config['datasets'],'splits:',config['splits'],'arms:',config['llm_arms'])

## 3. Chạy preparation — CPU/RAM

Đọc embedding cache, fit whitening trên train, apply head có sẵn, common top50 → top5 → graph → lexical/AST → prompt. Stage này không gọi inference.

Thanh tiến độ cập nhật theo từng graph hoàn tất; hậu tố hiển thị cache hit, graph parse mới, số graph đang parse và ID mẫu đang chạy. Khi Colab parse mẫu lâu, số đang parse sẽ giữ nguyên trong lúc tổng tiến độ tạm đứng. Checkpoint cache được ghi theo mốc 500 graph nếu có graph mới và sau mỗi arm. Nếu bấm Interrupt, finally cố ghi thêm checkpoint. Nếu runtime bị xóa đột ngột, dùng checkpoint Drive cuối cùng ở lần chạy sau; cache có trong checkpoint được tái sử dụng. Không chạy hai notebook đồng thời với cùng TRANSFER.

In [ ]:
from grace_h1.experiment import prepare_all
from grace_h1 import pipeline as _pipeline
from grace_h1.graphs import GraphStore as _GraphStore
from concurrent.futures import ThreadPoolExecutor as _ThreadPoolExecutor
from threading import Lock as _Lock, Event as _Event, Thread as _Thread
import importlib.util
if importlib.util.find_spec('tqdm') is None: subprocess.check_call([sys.executable,'-m','pip','install','tqdm>=4.66'])
from tqdm.auto import tqdm
import pandas as pd

# Theo dõi runtime trong notebook mà không đổi source fingerprint của CPU checkpoint.
_progress_lock=_Lock();_progress={'cache':0,'parsed':0};_parsing={}
_original_ensure=_GraphStore.ensure
def _tracked_ensure(self,row):
    h=row['code_sha256'];cached=self.exists(h)
    if not cached:
        with _progress_lock: _parsing[h]=str(row.get('record_id',h[:12]))
    try:
        result=_original_ensure(self,row)
        with _progress_lock: _progress['cache' if cached else 'parsed']+=1
        return result
    finally:
        if not cached:
            with _progress_lock: _parsing.pop(h,None)
_GraphStore.ensure=_tracked_ensure
class _ProgressPool(_ThreadPoolExecutor):
    def map(self,fn,*iterables,timeout=None,chunksize=1):
        total=len(iterables[0]) if iterables else None
        with _progress_lock: _progress.update(cache=0,parsed=0);_parsing.clear()
        results=super().map(fn,*iterables,timeout=timeout,chunksize=chunksize)
        def _with_progress():
            with tqdm(total=total,desc='Graph preparation',unit='graph',dynamic_ncols=True) as bar:
                stop=_Event()
                def _refresh():
                    while not stop.wait(2):
                        with _progress_lock:
                            cache=_progress['cache'];parsed=_progress['parsed'];active=list(_parsing.values())
                        bar.set_postfix_str(f'cache {cache} | parsed {parsed} | đang parse {len(active)}: {", ".join(active)[:70] or "-"}')
                        bar.refresh()
                monitor=_Thread(target=_refresh,daemon=True);monitor.start()
                try:
                    for result in results:
                        bar.update(1)
                        with _progress_lock:
                            cache=_progress['cache'];parsed=_progress['parsed'];active=list(_parsing.values())
                        bar.set_postfix_str(f'cache {cache} | parsed {parsed} | đang parse {len(active)}: {", ".join(active)[:70] or "-"}')
                        yield result
                finally:
                    stop.set();monitor.join(timeout=3)
        return _with_progress()
_pipeline.ThreadPoolExecutor=_ProgressPool

def save_checkpoint():
    info=create_bundle(ROOT,ROOT/'bundles'/'grace-h1-cpu-checkpoint.zip','cpu_checkpoint')
    temp=CHECKPOINT.with_name(CHECKPOINT.name+'.tmp')
    shutil.copyfile(info['path'],temp);os.replace(temp,CHECKPOINT)
    tqdm.write(f'Checkpoint saved: {CHECKPOINT} bytes: {info["bytes"]}')

try:
    runs=prepare_all(ROOT,config,checkpoint_callback=save_checkpoint)
finally:
    save_checkpoint()
display(pd.DataFrame(runs))

## 4. Xuất kết quả hoàn tất và quay về local

cpu-prepared chứa prompts/retrieval/freeze/run selection, vector/candidate cache mới và graph mới phát sinh. Không đóng gói lại graph gốc đã nằm trong input; không có response hay credential. Notebook kiểm tra đủ mọi arm trước khi xuất.

Tải grace-h1-cpu-prepared.zip về bundles/ của project local. Giữ grace-h1-trained-heads.zip tại đó. Quay lại full-pipeline.ipynb và Run All: notebook nhận các bundle, bỏ qua training/preparation local, rồi mới preflight → gọi Vertex → evaluation.

In [ ]:
from grace_h1.offload import require_prepared_runs
print('Frozen runs ready:',require_prepared_runs(ROOT,config))
info=create_bundle(ROOT,ROOT/'bundles'/'grace-h1-cpu-prepared.zip','cpu_prepared')
temp=OUTPUT.with_name(OUTPUT.name+'.tmp')
shutil.copyfile(info['path'],temp);os.replace(temp,OUTPUT)
print('DOWNLOAD THIS:',OUTPUT,'bytes:',info['bytes'],'sha256:',info['sha256'])
from IPython.display import FileLink,display
display(FileLink(str(OUTPUT)))
# If USE_GOOGLE_DRIVE=False, download from the Files pane before disconnecting.